In [1]:
import json
import numpy as np
from pandas import read_csv
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
import joblib


In [2]:
#Variables generales
ruta_model = "../../Model/"
ruta_metrics = "../../Metrics/"
ruta_df = "../../Datasets/"

semilla = 111

## FUNCIÓN BASE

In [3]:
def train_xgboost(X, y, output_model_path = 'best_model.pkl', output_metrics_path = 'metrics.json', kf = 2, param_grid = {'n_estimators': [50]}):
    # Initialize XGBoost Classifier
    xgb_model = xgb.XGBClassifier(use_label_encoder = False
                                  , eval_metric = 'mlogloss'
                                  , random_state = 42
                                  )
    
    # Define K-Fold Cross Validation
    kf = KFold(n_splits = kf
               , shuffle = True
               , random_state = 42)
    
    # Cross Validation
    grid_search = GridSearchCV(estimator = xgb_model
                               , param_grid = param_grid
                               , cv = kf
                               , scoring = 'accuracy'
                               , n_jobs = -1
                               , verbose = 2
                               )
    
    # Ajustar el modelo
    grid_search.fit(X, y)
    
    # mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guarde el mejor modelo en un archivo .pkl
    joblib.dump(best_model, output_model_path)
    
    # mejor modelo.
    y_pred = best_model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average = 'weighted'),
        'recall': recall_score(y, y_pred, average = 'weighted'),
        'f1_score': f1_score(y, y_pred, average = 'weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class = 'ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict = True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guarde las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)

    return metrics, best_model


## Entrenamiento

### df interpolation

* SMOTE

In [4]:
# carga de caracteristicas
df_IM_smote = read_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [5]:
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }

metric_1, model_1 = train_xgboost(X = df_IM_smote.drop(columns='FLAG')
                                  , y = df_IM_smote['FLAG']
                                  , output_model_path = '{}XGB_IM_S.pkl'.format(ruta_model)
                                  , output_metrics_path = '{}XGB_IM_S.json'.format(ruta_metrics)
                                  , kf = 3
                                  , param_grid = param_grid)

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [6]:
metric_1

{'accuracy': 0.89877582718522,
 'precision': 0.898799184815171,
 'recall': 0.89877582718522,
 'f1_score': 0.8987866690679156,
 'roc_auc': 0.8976184851513418,
 'confusion_matrix': [[24684, 2500], [2453, 19294]],
 'classification_report': {'0': {'precision': 0.9096068098905553,
   'recall': 0.9080341377280753,
   'f1-score': 0.9088197934500468,
   'support': 27184},
  '1': {'precision': 0.8852895292282279,
   'recall': 0.887202832574608,
   'f1-score': 0.8862451482510737,
   'support': 21747},
  'accuracy': 0.89877582718522,
  'macro avg': {'precision': 0.8974481695593917,
   'recall': 0.8976184851513416,
   'f1-score': 0.8975324708505603,
   'support': 48931},
  'weighted avg': {'precision': 0.898799184815171,
   'recall': 0.89877582718522,
   'f1-score': 0.8987866690679156,
   'support': 48931}},
 'balanced_accuracy': 0.8976184851513416,
 'cohen_kappa': 0.7950651331026548,
 'matthews_corrcoef': 0.7950666364686143}

* ADASYN

In [7]:
# carga de caracteristicas
df_IM_adasyn = read_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [8]:
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }

metric_2, model_2 = train_xgboost(X = df_IM_adasyn.drop(columns='FLAG')
                                  , y = df_IM_adasyn['FLAG']
                                  , output_model_path = '{}XGB_IM_A.pkl'.format(ruta_model)
                                  , output_metrics_path = '{}XGB_IM_A.json'.format(ruta_metrics)
                                  , kf = 3
                                  , param_grid = param_grid)

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [9]:
metric_2

{'accuracy': 0.8791469679294325,
 'precision': 0.8792895115423692,
 'recall': 0.8791469679294325,
 'f1_score': 0.8792026442495453,
 'roc_auc': 0.8780709792940784,
 'confusion_matrix': [[24129, 3055], [2850, 18827]],
 'classification_report': {'0': {'precision': 0.8943622817747137,
   'recall': 0.8876177163037081,
   'f1-score': 0.8909772353820874,
   'support': 27184},
  '1': {'precision': 0.8603875331322548,
   'recall': 0.8685242422844489,
   'f1-score': 0.8644367409720148,
   'support': 21677},
  'accuracy': 0.8791469679294325,
  'macro avg': {'precision': 0.8773749074534842,
   'recall': 0.8780709792940785,
   'f1-score': 0.8777069881770512,
   'support': 48861},
  'weighted avg': {'precision': 0.8792895115423692,
   'recall': 0.8791469679294325,
   'f1-score': 0.8792026442495453,
   'support': 48861}},
 'balanced_accuracy': 0.8780709792940785,
 'cohen_kappa': 0.7554183329874425,
 'matthews_corrcoef': 0.7554455660653546}

### df linear regression

* SMOTE

In [10]:
# carga de caracteristicas
df_LR_smote = read_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [11]:
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }

metric_3, model_3 = train_xgboost(X = df_LR_smote.drop(columns='FLAG')
                                  , y = df_LR_smote['FLAG']
                                  , output_model_path = '{}XGB_LR_S.pkl'.format(ruta_model)
                                  , output_metrics_path = '{}XGB_LR_S.json'.format(ruta_metrics)
                                  , kf = 3
                                  , param_grid = param_grid)

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [12]:
metric_3

{'accuracy': 0.930228280640085,
 'precision': 0.9305286995000175,
 'recall': 0.930228280640085,
 'f1_score': 0.9302956951333127,
 'roc_auc': 0.9304137180077716,
 'confusion_matrix': [[25247, 1937], [1477, 20270]],
 'classification_report': {'0': {'precision': 0.944731327645562,
   'recall': 0.9287448499117128,
   'f1-score': 0.9366698820212214,
   'support': 27184},
  '1': {'precision': 0.9127752510469672,
   'recall': 0.9320825861038304,
   'f1-score': 0.9223278882468035,
   'support': 21747},
  'accuracy': 0.930228280640085,
  'macro avg': {'precision': 0.9287532893462647,
   'recall': 0.9304137180077716,
   'f1-score': 0.9294988851340125,
   'support': 48931},
  'weighted avg': {'precision': 0.9305286995000175,
   'recall': 0.930228280640085,
   'f1-score': 0.9302956951333127,
   'support': 48931}},
 'balanced_accuracy': 0.9304137180077716,
 'cohen_kappa': 0.8590103610075186,
 'matthews_corrcoef': 0.8591654028779037}

* ADASYN

In [13]:
# carga de caracteristicas
df_LR_adasyn = read_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [14]:
param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9]
    }

metric_4, model_4 = train_xgboost(X = df_LR_adasyn.drop(columns='FLAG')
                                  , y = df_LR_adasyn['FLAG']
                                  , output_model_path = '{}XGB_LR_A.pkl'.format(ruta_model)
                                  , output_metrics_path = '{}XGB_LR_A.json'.format(ruta_metrics)
                                  , kf = 3
                                  , param_grid = param_grid)

Fitting 3 folds for each of 243 candidates, totalling 729 fits


In [15]:
metric_4

{'accuracy': 0.8904179932217315,
 'precision': 0.8909144601067527,
 'recall': 0.8904179932217315,
 'f1_score': 0.8905506227983133,
 'roc_auc': 0.8902484352740592,
 'confusion_matrix': [[24240, 2944], [2391, 19110]],
 'classification_report': {'0': {'precision': 0.9102174157936239,
   'recall': 0.8917010005885815,
   'f1-score': 0.9008640713555701,
   'support': 27184},
  '1': {'precision': 0.8665094767389135,
   'recall': 0.8887958699595367,
   'f1-score': 0.8775111927448054,
   'support': 21501},
  'accuracy': 0.8904179932217315,
  'macro avg': {'precision': 0.8883634462662687,
   'recall': 0.8902484352740592,
   'f1-score': 0.8891876320501877,
   'support': 48685},
  'weighted avg': {'precision': 0.8909144601067527,
   'recall': 0.8904179932217315,
   'f1-score': 0.8905506227983133,
   'support': 48685}},
 'balanced_accuracy': 0.8902484352740592,
 'cohen_kappa': 0.7784041755902568,
 'matthews_corrcoef': 0.7786095997945376}